# 00. Data Preprocessing: The Builder Pipeline

**Project:** Synthetic Raman Spectra Generation for Glioma Detection  
**Role:** Builder (Raw Data $\rightarrow$ Processed HDF5)

This notebook executes the **one-time** preprocessing pipeline required to convert the raw 2D spectral scans into a machine-learning-ready dataset. Due to the computational cost of baseline correction algorithms, we separate this "Building" phase from the "Training" phase.

### 🎯 Objectives
1.  **Ingest:** Load raw `.npy` spectral cubes from the `data/raw/` directory.
2.  **Align:** Match spectra with their corresponding ground-truth tumor maps (`data/tumor_maps.npy`).
3.  **Clean:** Apply baseline correction and normalization to remove non-biological noise.
4.  **Structure:** Save the processed dataset into a highly optimized Hierarchical Data Format (HDF5) file for efficient I/O during training.

## 🔬 Preprocessing Methodology

Raman spectra are inherently noisy and contain background fluorescence that must be removed before training deep learning models. This pipeline applies the following transformations:

### 1. Baseline Correction: AirPLS
We utilize **Adaptive Iteratively Reweighted Penalized Least Squares (AirPLS)**. 
* **Purpose:** Removes the sloping background fluorescence (baseline drift) that varies between samples and spectrometers, while preserving the sharp Raman peaks that correspond to molecular vibrations.
* **Parameters:** `lambda=4`, `porder=1`.

### 2. Normalization: Min-Max
* **Purpose:** Scales the intensity of each spectrum to the range `[0, 1]`.
* **Why:** Neural networks (like VAEs and CNNs) converge significantly faster and more stably when input features are normalized.

### 3. Quality Control
* **Exclusion:** The sample `HF-1887_via-t_2_1.h5_3` is explicitly excluded from the dataset (as per previous literature, see APOLLO project) due to data quality issues.
* **Clipping:** Negative values resulting from baseline subtraction are clipped to 0.

In [1]:
# 00_Preprocessing.ipynb

import sys
import os
from pathlib import Path

# Add project root to path so we can import 'src'
# (Assuming notebook is in /notebooks and src is in /src)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.preprocessing import RamanPreprocessor
from src.data.loader import GliomaDataLoader
import numpy as np
import matplotlib.pyplot as plt

# --- Configuration ---
RAW_DATA_DIR = Path("../data/raw")
TUMOR_MAPS_PATH = Path("../data/tumor_maps.npy")
OUTPUT_FILE = Path("../data/processed/glioma_dataset.h5")

# --- Step 1: Initialize the Builder ---
# We use the class we verified in the test script
preprocessor = RamanPreprocessor(
    raw_data_dir=RAW_DATA_DIR,
    tumor_maps_path=TUMOR_MAPS_PATH,
    output_file=OUTPUT_FILE
)

print(f"Source: {preprocessor.raw_data_dir}")
print(f"Target: {preprocessor.output_file}")

2026-01-20 09:26:00.343985: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-20 09:26:03.088085: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Source: ../data/raw
Target: ../data/processed/glioma_dataset.h5


> ⚠️ **IMPORTANT USAGE NOTE**
>
> **Run this notebook only ONCE.**
>
> The AirPLS algorithm is computationally intensive. Processing ~300,000 spectra may take **1-2 hours** depending on your CPU. Once the `glioma_dataset.h5` file is generated, you do not need to run this notebook again unless the raw data changes.
>
> Future experiments (CVAE, SVM, etc.) will simply load the processed `.h5` file in seconds.

In [ ]:
# --- Step 2: Run the Pipeline ---
# This will take time (approx 1-2 hours for 300k spectra due to airPLS)
# The progress bar will show the status.
preprocessor.run()

Scanning ../data/raw...
Loading tumor maps from ../data/tumor_maps.npy...
Loaded 59 maps for 60 files.
Starting Streaming Process...


  0%|          | 0/59 [00:00<?, ?it/s]

## 📂 Output Data Structure

The pipeline generates a single file: `data/processed/glioma_dataset.h5`. This file acts as a structured database containing:

| Dataset Key | Shape | Dtype | Description |
| :--- | :--- | :--- | :--- |
| **`/spectra`** | `(N, Bands)` | `float32` | The cleaned, normalized 1D Raman spectra. |
| **`/y_class`** | `(N,)` | `int32` | The specific glioma sub-type (Classes 1-6). |
| **`/y_tumor`** | `(N,)` | `int32` | Binary label: `0` = Normal Tissue, `1` = Tumor. |
| **`/sample_ids`** | `(N,)` | `string` | **CRITICAL:** The unique ID of the tissue slice (e.g., `HF-448`). Used to ensure strict Train/Test separation. |
| **`/patient_ids`**| `(N,)` | `int32` | The ID of the patient. |

This structure supports **"Group-Aware Splitting"**: when we train models later, we can split data based on `/sample_ids` to ensure that pixels from the same tissue slice never leak between Training and Testing sets.

In [ ]:
# --- Step 3: Verify the Output (Sanity Check) ---
# Load the file we just created to ensure it's readable
loader = GliomaDataLoader(OUTPUT_FILE, memory_mapped=True)

print(f"Total Spectra: {loader.X.shape[0]}")
print(f"Total Bands: {loader.X.shape[1]}")
print(f"Unique Samples: {len(np.unique(loader.groups))}")

# Visual Sanity Check
# Plot mean spectrum of Tumor vs Non-Tumor to see if biology is preserved
plt.figure(figsize=(10, 5))

# Indices for tumor vs normal
idx_tumor = np.where(loader.y_tumor[:] == 1)[0]
idx_normal = np.where(loader.y_tumor[:] == 0)[0]

# Compute mean (using a subset to save RAM if needed)
mean_tumor = np.mean(loader.X[idx_tumor[:5000]], axis=0)
mean_normal = np.mean(loader.X[idx_normal[:5000]], axis=0)

plt.plot(mean_tumor, color='red', label='Tumor (Mean)')
plt.plot(mean_normal, color='green', label='Normal (Mean)')
plt.title("Sanity Check: Average Spectra")
plt.legend()
plt.show()

In [ ]:
from src.data.splitter import Splitter
# --- Create Data Splits ---
# 2. Generate and Save Splits
splitter = Splitter(data_path="../data/processed/glioma_dataset.h5", output_dir="../data/splits")
splitter.generate_manual_split() # Generates manual_splits.json